# Tarea puente — Guía L2: El índice sancocho

**Cálculo Diferencial — Laboratorio computacional** · Universidad Nacional de Colombia, Sede Manizales

Responsable: Thomas Ramírez Villa. Equipo: Nicol Velez Arboleda, Sofía Hoyos Tabares, Sebastián Gutiérrez Rodríguez y Thomas Ramírez Villa.


In [ ]:
import pandas as pd

# Congelado del curso (el mismo que usan los cuadernos de las sesiones)
RUTA = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/datos/sipsa_mensual.csv.gz"
df = pd.read_csv(RUTA)
print(df.shape)

## Verificación del congelado: verificación del congelado (Thomas)

Se clona el repositorio del curso y se reconstruye el congelado desde los ocho ZIP oficiales del DANE (2013–2017 y uno por año de 2018 a 2024). Después se compara con lo que dice `datos/README.md`: 666.865 filas, 144 meses (2013-01 a 2024-12), 71 ciudades, 385 productos y el plátano dominico hartón verde en Manizales, Centro Galerías, a \$2.663/kg en 2024-12. La descarga tarda unos minutos.

**Relato de la diferencia (5 de noviembre de 2026):** el clon del repositorio funcionó, pero `congelar_sipsa.py` se detuvo en la primera descarga. El portal de microdatos del DANE respondió `HTTP Error 401: Unauthorized`, porque desde Colab, sin una sesión iniciada en el portal, no deja bajar los ZIP. Por eso la celda de abajo revisa los mismos controles sobre el congelado del curso: 666.865 filas, 144 meses, 71 ciudades, 385 productos y el plátano a \$2.663 en 2024-12, y todos coinciden. Esto confirma que el congelado del curso cumple lo que dice el README, pero **no es una reproducción independiente desde la fuente**. Para reproducirlo habría que iniciar sesión en el portal de microdatos del DANE, bajar los ocho ZIP a mano y adaptar el script para que los lea desde disco.


In [ ]:
import os, subprocess
VERIFICAR_CONGELADO = True   # descarga ~8 ZIP del DANE; si el DANE está caído, el cuaderno sigue corriendo
try:
    if not os.path.exists("labs-calculo-2026-2"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/cmorregof/labs-calculo-2026-2"], check=True)
    if VERIFICAR_CONGELADO and not os.path.exists("/tmp/sipsa_mensual.csv.gz"):
        r = subprocess.run(["python3", "datos/congelar_sipsa.py", "/tmp/sipsa_mensual.csv.gz"],
                           cwd="labs-calculo-2026-2", capture_output=True, text=True)
        print(r.stdout[-1500:])
        if r.returncode != 0:
            raise RuntimeError(r.stderr.strip().splitlines()[-1] if r.stderr.strip() else f"código {r.returncode}")
    nuevo = pd.read_csv("/tmp/sipsa_mensual.csv.gz")
    reconstruido = True
except Exception as e:
    print("⚠️ No se pudo reconstruir el congelado desde el DANE:", e)
    print("   Se revisan los mismos números sobre el congelado del curso (df). Esto NO es la reproducción;")
    print("   para la entrega hay que volver a correr esta celda cuando el servidor del DANE responda.")
    nuevo = df.copy()
    reconstruido = False

control = nuevo[(nuevo["producto"] == "Plátano dominico hartón verde")
                & (nuevo["mercado"] == "Manizales, Centro Galerías") & (nuevo["fecha"] == "2024-12")]["precio_kg"]
chequeos = {
    "filas = 666 865":         len(nuevo) == 666_865,
    "meses = 144":             nuevo["fecha"].nunique() == 144,
    "2013-01 a 2024-12":       (nuevo["fecha"].min(), nuevo["fecha"].max()) == ("2013-01", "2024-12"),
    "ciudades = 71":           nuevo["ciudad"].nunique() == 71,
    "productos = 385":         nuevo["producto"].nunique() == 385,
    "plátano MZL 2024-12 = 2663": len(control) == 1 and round(control.iloc[0]) == 2663,
}
for nombre, ok in chequeos.items():
    print(("✅" if ok else "❌"), nombre)
print(f"\nFilas: {len(nuevo):,} · meses: {nuevo['fecha'].nunique()} · ciudades: {nuevo['ciudad'].nunique()} · "
      f"productos: {nuevo['producto'].nunique()} · plátano: ${control.iloc[0]:,.0f}")
iguales = nuevo.sort_values(list(nuevo.columns[:4])).reset_index(drop=True).equals(
          df.sort_values(list(df.columns[:4])).reset_index(drop=True))
print("¿El congelado reconstruido es idéntico al del curso?", iguales if reconstruido else "no aplica (no se reconstruyó)")
# 📸 Tomar pantallazo de esta salida para la entrega.

Descargando 2013-2017…

⚠️ No se pudo reconstruir el congelado desde el DANE: urllib.error.HTTPError: HTTP Error 401: Unauthorized
   Se revisan los mismos números sobre el congelado del curso (df). Esto NO es la reproducción;
   para la entrega hay que volver a correr esta celda cuando el servidor del DANE responda.
✅ filas = 666 865
✅ meses = 144
✅ 2013-01 a 2024-12
✅ ciudades = 71
✅ productos = 385
✅ plátano MZL 2024-12 = 2663

Filas: 666,865 · meses: 144 · ciudades: 71 · productos: 385 · plátano: $2,663
¿El congelado reconstruido es idéntico al del curso? no aplica (no se reconstruyó)


**Defensa del precio medio (Thomas):**

1. El precio promedio por kilo es una buena medida de la **tendencia** del costo de la comida, porque es estable, se reporta todos los meses y permite comparar años y ciudades. No mide el costo de vida de una familia: es precio **mayorista** de una central, no lo que se paga en la tienda.
2. Pagan más que ese promedio quienes compran al detal y en pocas cantidades (tienda de barrio, minimercado), quienes viven lejos de la central (zona rural, barrios periféricos, con transporte e intermediarios encima) y quienes no pueden comprar cuando el precio baja ni guardar comida.
3. Además, la familia gasta en lo que el índice no cuenta: cilantro, cebolla, ajo, gas, arriendo y transporte.
4. Para un índice del **precio máximo** o de la **mediana** harían falta las cotizaciones individuales de cada día y de cada vendedor (los microdatos de SIPSA), no el promedio mensual ya agregado que trae el congelado.
5. Para un índice **minorista** harían falta precios de tienda y supermercado por barrio y ciudad, el tamaño de los empaques y las cantidades que compra realmente cada hogar.
